# Check the 2024 WASH priority shortlist

This notebook uses the earlier state results to check how the priority list changes at **40%, 50% and 60%**. It also shows the separate effect of missing outcomes. In the final analysis section, it uses the household data to estimate which WASH domains have the largest service gaps among the states above the main cutoff.

1. Put the notebook in the same JupyterLab folder as **wash_state_priority_outputs** or place **State_Priority_Estimates_2024.csv** next to it. Run the earlier state-priority notebook first if you do not have the CSV.
2. Put **Nigeria_DHS_2013_2018_2024_Households_Climate.csv.gz** next to the notebook or in **climate_stack_outputs**. Leave it compressed. The threshold analysis uses the CSV only. The domain profiles need the household file. If your file has a different name, change `HOUSEHOLD_FILE` in the setup cell.
3. Select the **Python 3** kernel. If a package is missing, run `%pip install pandas numpy matplotlib pillow python-docx` in a separate cell, then restart the kernel.
4. Select **Run → Run All Cells**. Results appear in **wash_priority_sensitivity_outputs**. The model is not refitted. Only the 2024 survey round is used in this follow-up.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import os
import shutil
import subprocess

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

ROOT = Path.cwd()
OUTPUT = ROOT / "wash_priority_sensitivity_outputs"
STATE_FILE = ROOT / "wash_state_priority_outputs" / "State_Priority_Estimates_2024.csv"
if not STATE_FILE.is_file():
    STATE_FILE = ROOT / STATE_FILE.name

HOUSEHOLD_FILE = ROOT / "Nigeria_DHS_2013_2018_2024_Households_Climate.csv.gz"
if not HOUSEHOLD_FILE.is_file():
    HOUSEHOLD_FILE = ROOT / "climate_stack_outputs" / HOUSEHOLD_FILE.name

CUTOFFS = (40, 50, 60)
MAIN = 50
ABOVE, UNCERTAIN, BELOW = "Above", "Uncertain", "Below"
if not STATE_FILE.is_file():
    raise FileNotFoundError("Run the state-priority notebook to generate State_Priority_Estimates_2024.csv.")
OUTPUT.mkdir(exist_ok=True)

## 1. Create the decision table

For a cutoff of 50%, a state is **Above** when its lower 95% survey confidence limit exceeds 50%. It is **Below** when its upper limit is under 50%. Otherwise it is **Uncertain**. We apply the same rule at 40% and 60%.

The missing-outcome bounds come from the earlier notebook. The lower bound treats every missing outcome as non-deprived. The upper bound treats every missing outcome as deprived. If these bounds touch or cross 50%, flag that state for review. These bounds are a separate sensitivity check and do not include sampling uncertainty.

The main shortlist contains all states classified Above at 50%. Within it, the strict shortlist requires **Above at all three cutoffs** and a **missing-outcome lower bound above 50%**. This deliberately conservative secondary shortlist gives you a way to distinguish the most stable decisions under the checks used here. The model flag is recomputed from its saved predicted percentage using the same 50% cutoff.

In [ ]:
states = pd.read_csv(STATE_FILE)
required = ["state", "zone", "all_n", "eligible_n", "weighted_missing_pct",
            "observed_pct", "observed_ci95_low_pct", "observed_ci95_high_pct",
            "model_pct", "missing_lower_pct", "missing_upper_pct"]
if missing := set(required) - set(states):
    raise ValueError(f"State results are missing columns: {sorted(missing)}.")
states = states[required].copy()
if len(states) != 37 or states.state.isna().any() or states.state.duplicated().any() or "FCT" not in set(states.state):
    raise ValueError("Expected one row for each of the 36 states and the FCT.")
pct_columns = required[4:]
if not states[pct_columns].apply(lambda column: column.between(0, 100)).all().all():
    raise ValueError("The required percentage values must be on a 0–100 scale.")
if (states.observed_ci95_low_pct > states.observed_ci95_high_pct).any():
    raise ValueError("A survey interval has reversed limits.")
if ((states.missing_lower_pct > states.observed_pct + 1e-8) |
    (states.missing_upper_pct < states.observed_pct - 1e-8)).any():
    raise ValueError("A missing-outcome bound is inconsistent with the point estimate.")
if not np.allclose(states.missing_upper_pct - states.missing_lower_pct,
                   states.weighted_missing_pct, atol=1e-5, rtol=0):
    raise ValueError("Missing-outcome bounds do not match the weighted missing percentage.")

for cutoff in CUTOFFS:
    states[f"survey_class_{cutoff}"] = np.select(
        [states.observed_ci95_low_pct.gt(cutoff), states.observed_ci95_high_pct.lt(cutoff)],
        [ABOVE, BELOW], default=UNCERTAIN)

main_class = f"survey_class_{MAIN}"
states["above_all_three_cutoffs"] = states[[f"survey_class_{k}" for k in CUTOFFS]].eq(ABOVE).all(axis=1)
states["classification_changes_across_cutoffs"] = (
    states[[f"survey_class_{k}" for k in CUTOFFS]].nunique(axis=1).gt(1))
states["missing_bounds_touch_or_cross_50"] = (
    states.missing_lower_pct.le(MAIN) & states.missing_upper_pct.ge(MAIN))
states["strict_shortlist"] = states.above_all_three_cutoffs & states.missing_lower_pct.gt(MAIN)
states["model_flags_at_50"] = states.model_pct.ge(MAIN)
states["model_bias_pp"] = states.model_pct - states.observed_pct
states["survey_model_comparison_at_50"] = np.select(
    [states[main_class].eq(UNCERTAIN),
     states[main_class].eq(ABOVE) & ~states.model_flags_at_50,
     states[main_class].eq(BELOW) & states.model_flags_at_50],
    ["Survey uncertain", "Model misses survey priority", "Model flags survey-below area"],
    default="Agreement")
states = states.sort_values(["observed_pct", "state"], ascending=[False, True])
main_shortlist = states.loc[states[main_class].eq(ABOVE)].copy()
strict_shortlist = states.loc[states.strict_shortlist].copy()
missing_review = states.loc[states.missing_bounds_touch_or_cross_50].copy()
counts = pd.DataFrame([
    {"threshold_pct": k, **{label: int(states[f"survey_class_{k}"].eq(label).sum())
                           for label in (ABOVE, UNCERTAIN, BELOW)}}
    for k in CUTOFFS
])
for name, table in {
    "State_Priority_Decision_Table_2024.csv": states,
    "State_Priority_Threshold_Counts_2024.csv": counts,
    "State_Priority_Main_Shortlist_2024.csv": main_shortlist,
    "State_Priority_Strict_Shortlist_2024.csv": strict_shortlist,
    "State_Priority_Missing_Review_2024.csv": missing_review,
}.items():
    table.to_csv(OUTPUT / name, index=False)

display(counts)
print(f"Main shortlist: {len(main_shortlist)}. Strict shortlist: {len(strict_shortlist)}.")
print("Strict shortlist:", ", ".join(strict_shortlist.state) or "None")
print("Missing-outcome review:", ", ".join(missing_review.state) or "None")
display(main_shortlist[["state", "observed_pct", "survey_class_40", "survey_class_50",
                        "survey_class_60", "missing_bounds_touch_or_cross_50",
                        "survey_model_comparison_at_50"]].round(1))

## 2. Profile the water, sanitation and hygiene gaps

Each deprivation variable uses 1 for deprived, 0 for non-deprived and blank for unknown. The primary estimate for each domain uses households with a known value in that domain. The denominator can differ across domains, so the table also shows the estimate on the common set of households with all three outcomes known. Compare those estimates alongside the weighted missing percentage, especially where coverage differs by domain.

The intervals are pointwise and use a stratified PSU bootstrap of the full 2024 survey design. They quantify sampling variation among households with known domain values. The domain estimates describe service gaps and can overlap within the same household. The CSV contains profiles for the entire main shortlist and identifies the strict subset.

In [ ]:
if not HOUSEHOLD_FILE.is_file():
    raise FileNotFoundError("The decision CSVs are saved. Add the harmonised household file to run the domain profiles.")

DOMAINS = {"water_dep": "Drinking water", "sanitation_dep": "Sanitation", "hygiene_dep": "Hygiene"}
PSU, STRATUM, WEIGHT = "psu_key", "stratum_hv023_key", "hh_weight"
TARGET = "wash_dep_2plus"
cols = ["survey_round", "hv001", "hv002", "state", WEIGHT, PSU, STRATUM, TARGET, *DOMAINS]
house = pd.read_csv(HOUSEHOLD_FILE, usecols=cols)
house = house.loc[house.survey_round.eq(2024)].copy()
if len(house) != int(states.all_n.sum()) or set(house.state) != set(states.state):
    raise ValueError("Household totals or state names differ from the state results.")
if house[["hv001", "hv002", PSU, STRATUM]].isna().any().any() or house.duplicated(["hv001", "hv002"]).any():
    raise ValueError("Missing survey identifiers or duplicate household keys.")
if not np.isfinite(house[WEIGHT]).all() or not house[WEIGHT].gt(0).all():
    raise ValueError("Survey weights must be positive and finite.")
for col in [TARGET, *DOMAINS]:
    if not house[col].dropna().isin([0, 1]).all():
        raise ValueError(f"{col} must contain only 0, 1 or missing values.")
known = house[list(DOMAINS)].notna().all(axis=1)
if not known.eq(house[TARGET].notna()).all():
    raise ValueError("The composite indicator is expected on households with all three domain outcomes known.")
derived = house.loc[known, list(DOMAINS)].sum(axis=1).ge(2).astype(int)
if not derived.eq(house.loc[known, TARGET]).all():
    raise ValueError("The household composite outcome differs from the 2-of-3 rule.")

check = states.set_index("state")
for state, group in house.groupby("state"):
    eligible = group.loc[group[TARGET].notna()]
    observed = 100 * np.average(eligible[TARGET], weights=eligible[WEIGHT])
    if (len(group) != check.loc[state, "all_n"] or len(eligible) != check.loc[state, "eligible_n"]
            or not np.isclose(observed, check.loc[state, "observed_pct"], atol=1e-5, rtol=0)):
        raise ValueError(f"Household counts or observed prevalence differ from the state results for {state}.")

design = house[[PSU, STRATUM]].drop_duplicates().sort_values([STRATUM, PSU]).reset_index(drop=True)
if (design[PSU].duplicated().any() or design.groupby(STRATUM).size().lt(2).any()
        or house.groupby(PSU).state.nunique().ne(1).any()):
    raise ValueError("Check the 2024 PSU and stratum definitions before calculating intervals.")
rng = np.random.default_rng(20241112)
rep = np.zeros((999, len(design)))
for _, members in design.groupby(STRATUM, sort=False):
    n = len(members)
    rep[:, members.index] = rng.multinomial(n-1, np.full(n, 1/n), size=len(rep)) * n/(n-1)
psu_lookup = pd.Series(design.index, index=design[PSU])

profile_columns = ["state", "domain", "strict_shortlist", "all_n", "domain_known_n", "domain_psus",
                   "weighted_missing_pct", "deprived_pct", "ci95_low_pct", "ci95_high_pct",
                   "common_complete_n", "common_complete_deprived_pct", "valid_bootstrap_replicates"]
profile_rows = []
for state in main_shortlist.state:
    group = house.loc[house.state.eq(state)]
    complete = group.loc[group[TARGET].notna()]
    for col, label in DOMAINS.items():
        g = group.loc[group[col].notna()]
        codes = g[PSU].map(psu_lookup).to_numpy(dtype=int)
        weights = g[WEIGHT].to_numpy()
        values = g[col].to_numpy()
        denominator = rep @ np.bincount(codes, weights=weights, minlength=len(design))
        numerator = rep @ np.bincount(codes, weights=weights * values, minlength=len(design))
        valid = denominator > 0
        lo, hi = (np.quantile(100 * numerator[valid] / denominator[valid], [.025, .975])
                  if valid.all() else (np.nan, np.nan))
        profile_rows.append({
            "state": state, "domain": label, "strict_shortlist": bool(check.loc[state, "strict_shortlist"]),
            "all_n": len(group), "domain_known_n": len(g), "domain_psus": g[PSU].nunique(),
            "weighted_missing_pct": 100 * group.loc[group[col].isna(), WEIGHT].sum() / group[WEIGHT].sum(),
            "deprived_pct": 100 * np.average(values, weights=weights),
            "ci95_low_pct": lo, "ci95_high_pct": hi,
            "common_complete_n": len(complete),
            "common_complete_deprived_pct": 100 * np.average(complete[col], weights=complete[WEIGHT]),
            "valid_bootstrap_replicates": int(valid.sum()),
        })
profiles = pd.DataFrame(profile_rows, columns=profile_columns)
profiles.to_csv(OUTPUT / "State_Priority_WASH_Domain_Profiles_2024.csv", index=False)
display(profiles.head(12).round(1))

## 3. Save the figures and Word report

The first figure shows how the classification counts change as the cutoff increases. The second profiles the strict shortlist and uses the main shortlist if the strict group is empty. The full domain CSV always covers the main shortlist.

The code saves two 600 dpi JPEGs and an editable report. If LibreOffice Writer is installed, it also saves a Word 97–2003 .doc file. Otherwise, open the .docx in desktop Word and choose **Save As → Word 97–2003 Document (.doc)**.

In [ ]:
def save_jpeg(fig, name):
    path = OUTPUT / name
    fig.savefig(path, dpi=600, format="jpeg", facecolor="white",
                pil_kwargs={"quality": 95, "subsampling": 0})
    plt.close(fig)
    with Image.open(path) as image:
        if image.format != "JPEG" or image.info.get("dpi") != (600, 600):
            raise ValueError(f"JPEG resolution check failed for {path.name}.")
    return path

fig, ax = plt.subplots(figsize=(7.4, 3.0), layout="constrained")
positions = np.arange(len(counts))
for i, (label, color) in enumerate(zip((ABOVE, UNCERTAIN, BELOW), ("#285A79", "#D3A344", "#83909D"))):
    bars = ax.bar(positions + (i-1)*.24, counts[label], width=.22, color=color, label=label)
    ax.bar_label(bars, fontsize=9, padding=2)
ax.set(xticks=positions, xticklabels=[f"{k}%" for k in CUTOFFS],
       xlabel="Priority cutoff", ylabel="Number of areas", ylim=(0, int(counts.iloc[:, 1:].max().max())+6))
ax.set_title("Survey priority classification at three cutoffs", fontweight="bold")
ax.legend(frameon=False, ncols=3, loc="upper center")
ax.spines[["top", "right"]].set_visible(False)
threshold_figure = save_jpeg(fig, "Figure_1_Priority_Threshold_Sensitivity_600dpi.jpg")

plot_states = list(strict_shortlist.state if len(strict_shortlist) else main_shortlist.state)
plot_label = "Strict shortlist" if len(strict_shortlist) else "Main shortlist"
fig, ax = plt.subplots(figsize=(7.4, max(2.8, 1.6 + .30*len(plot_states))), layout="constrained")
if plot_states:
    for i, (domain, color) in enumerate(zip(DOMAINS.values(), ("#24547C", "#AF442B", "#237C64"))):
        g = profiles.loc[profiles.domain.eq(domain)].set_index("state").loc[plot_states]
        y = np.arange(len(plot_states)) + (i-1)*.22
        ax.hlines(y, g.ci95_low_pct, g.ci95_high_pct, color=color, linewidth=1)
        ax.scatter(g.deprived_pct, y, s=18, color=color, label=domain, zorder=3)
    ax.set(yticks=np.arange(len(plot_states)), yticklabels=plot_states, xlim=(0, 100),
           xlabel="Weighted household deprivation (%)", axisbelow=True)
    ax.invert_yaxis()
    ax.grid(axis="x", color="#DDE2E7")
    ax.legend(frameon=False, ncols=3, loc="lower center", bbox_to_anchor=(.5, 1.0))
    ax.spines[["top", "right"]].set_visible(False)
else:
    ax.text(.5, .5, "No area qualified for the main shortlist.", ha="center", transform=ax.transAxes)
    ax.set_axis_off()
fig.suptitle(f"WASH domain gaps in the {plot_label.lower()}", fontweight="bold")
domain_figure = save_jpeg(fig, "Figure_2_Priority_WASH_Domain_Gaps_600dpi.jpg")

In [ ]:
from docx import Document
from docx.enum.table import WD_TABLE_ALIGNMENT, WD_CELL_VERTICAL_ALIGNMENT
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from docx.shared import Inches, Pt, RGBColor

figure_width = 7
doc = Document()
doc.core_properties.title = "Sensitivity of Nigeria’s 2024 WASH priority shortlist"
doc.core_properties.author = ""
section = doc.sections[0]
section.page_width, section.page_height = Inches(8.5), Inches(11)
section.left_margin = section.right_margin = Inches(0.75)
section.top_margin, section.bottom_margin = Inches(0.7), Inches(0.7)

for style_name, size in [("Normal", 10.5), ("Title", 25), ("Heading 1", 17), ("Heading 2", 13), ("Caption", 9)]:
    style = doc.styles[style_name]
    style.font.name = "Calibri"
    style.font.size = Pt(size)
    style.font.color.rgb = RGBColor.from_string("000000")
    style.paragraph_format.space_before = Pt(5)
    style.paragraph_format.space_after = Pt(5)
    if style_name == "Title":
        border = style.element.get_or_add_pPr().find(qn("w:pBdr"))
        if border is not None:
            border.getparent().remove(border)

def xml(parent, tag, **attributes):
    element = OxmlElement(f"w:{tag}")
    for key, value in attributes.items():
        element.set(qn(f"w:{key}"), str(value))
    parent.append(element)
    return element

def add_table(caption, headers, rows, widths, text_columns, font_size=9.5):
    doc.add_paragraph(caption, style="Caption").paragraph_format.keep_with_next = True
    table = doc.add_table(rows=0, cols=len(headers))
    table.alignment = WD_TABLE_ALIGNMENT.CENTER
    table.autofit = False
    for column, width in zip(table.columns, widths):
        column.width = Inches(width)
    props = table._tbl.tblPr
    borders = xml(props, "tblBorders")
    for edge in ("top", "left", "bottom", "right", "insideH", "insideV"):
        xml(borders, edge, val="single", sz=4, color="D9D9D9")
    margins = xml(props, "tblCellMar")
    for edge, value in (("top", 16), ("bottom", 16), ("left", 85), ("right", 85)):
        xml(margins, edge, w=value, type="dxa")

    for i, values in enumerate([headers, *rows]):
        row = table.add_row()
        row_props = row._tr.get_or_add_trPr()
        xml(row_props, "cantSplit")
        if i == 0:
            xml(row_props, "tblHeader")
        for j, (cell, value, width) in enumerate(zip(row.cells, values, widths)):
            cell.width = Inches(width)
            cell.vertical_alignment = WD_CELL_VERTICAL_ALIGNMENT.CENTER
            if i == 0 or i % 2 == 0:
                xml(cell._tc.get_or_add_tcPr(), "shd", fill="203A50" if i == 0 else "F2F6F9")
            paragraph = cell.paragraphs[0]
            paragraph.alignment = WD_ALIGN_PARAGRAPH.LEFT if j in text_columns else WD_ALIGN_PARAGRAPH.CENTER
            paragraph.paragraph_format.space_after = Pt(0)
            paragraph.paragraph_format.space_before = Pt(0)
            paragraph.paragraph_format.line_spacing = 1.05
            run = paragraph.add_run(str(value))
            run.bold = i == 0
            run.font.size = Pt(font_size)
            if i == 0:
                run.font.color.rgb = RGBColor.from_string("FFFFFF")
    return table

def add_figure(path, caption, description):
    paragraph = doc.add_paragraph()
    paragraph.paragraph_format.space_before = Pt(7)
    paragraph.paragraph_format.keep_with_next = True
    paragraph.alignment = WD_ALIGN_PARAGRAPH.CENTER
    shape = paragraph.add_run().add_picture(str(path), width=Inches(figure_width))
    shape._inline.docPr.set("descr", description)
    doc.add_paragraph(caption, style="Caption")





doc.add_paragraph("Sensitivity of Nigeria’s 2024 WASH priority shortlist", style="Title")
doc.add_paragraph(
    "This analysis uses direct survey estimates for the 36 states and the FCT. The 50% cutoff "
    "is the main study rule. An area qualifies when its lower 95% survey confidence limit exceeds the cutoff."
)
add_table("Table 1. Number of areas by cutoff and survey classification",
          ["Cutoff", "Above", "Uncertain", "Below"],
          [[f"{int(k)}%", int(a), int(u), int(b)] for k, a, u, b in counts.itertuples(index=False, name=None)],
          [1.6, 1.8, 1.8, 1.8], {0})
doc.add_paragraph(
    f"The main shortlist contains {len(main_shortlist)} areas. The strict shortlist contains "
    f"{len(strict_shortlist)} areas above all three cutoffs with a missing-outcome lower bound above 50%. "
    f"Missing-outcome bounds touch or cross 50% in {len(missing_review)} areas."
)
doc.add_paragraph("Strict shortlist in descending order of observed prevalence")
doc.add_paragraph(", ".join(strict_shortlist.state) or "No area met the strict rule.")
borno = states.set_index("state").loc["Borno"]
doc.add_paragraph(
    f"In Borno, {borno.weighted_missing_pct:.1f}% of weighted households lacked a composite outcome. "
    f"The missing-outcome bounds were {borno.missing_lower_pct:.1f}% to {borno.missing_upper_pct:.1f}%. "
    "These bounds show sensitivity to missing outcomes separately from sampling uncertainty."
)
add_figure(threshold_figure, "Figure 1. Survey classifications at the three cutoffs.",
           "Grouped bar chart showing the number of areas above, uncertain and below each cutoff.")
doc.add_heading("Domain gaps", level=1).paragraph_format.page_break_before = True
add_figure(domain_figure, f"Figure 2. Domain deprivation estimates with pointwise 95% intervals. {plot_label}.",
           "Dot and interval chart of drinking water, sanitation and hygiene deprivation by state.")
doc.add_paragraph(
    "The plotted estimate for each domain uses the households with a known value for that domain. "
    "The domain CSV also gives estimates on a common set of households with all three outcomes known. "
    "Use the domain missingness columns when comparing services. The estimates describe 2024 conditions."
)
def find_soffice():
    custom = os.environ.get("LIBREOFFICE_PATH")
    if custom and Path(custom).is_file():
        return custom
    candidates = [shutil.which("soffice"), shutil.which("libreoffice"),
                  "/Applications/LibreOffice.app/Contents/MacOS/soffice"]
    for variable in ("ProgramFiles", "ProgramFiles(x86)"):
        if base := os.environ.get(variable):
            candidates.append(str(Path(base) / "LibreOffice" / "program" / "soffice.com"))
    return next((candidate for candidate in candidates if candidate and Path(candidate).is_file()), None)


docx_path = OUTPUT / "Nigeria_DHS_WASH_Priority_Sensitivity_Report.docx"
doc.save(docx_path)

def export_legacy_word(source, soffice):
    with TemporaryDirectory() as scratch:
        temp = Path(scratch).resolve()
        result = subprocess.run(
            [soffice, f"-env:UserInstallation={(temp / 'profile').as_uri()}", "--headless",
             "--convert-to", "doc:MS Word 97", "--outdir", str(temp), str(source.resolve())],
            capture_output=True, text=True, timeout=180,
        )
        converted = temp / source.with_suffix(".doc").name
        if result.returncode or not converted.is_file():
            raise RuntimeError(result.stderr or result.stdout or "Converter returned no file.")
        with converted.open("rb") as stream:
            if stream.read(8) != bytes.fromhex("D0CF11E0A1B11AE1"):
                raise ValueError("The converter did not create a binary Word 97–2003 document.")
        shutil.copyfile(converted, source.with_suffix(".doc"))

soffice = find_soffice()
if soffice:
    try:
        export_legacy_word(docx_path, soffice)
        print(f"Saved {docx_path.with_suffix('.doc').resolve()}")
    except (RuntimeError, ValueError, OSError, subprocess.TimeoutExpired) as error:
        print(f"Word .doc conversion failed: {error}")
        print("Open the saved .docx in desktop Word and save it as Word 97–2003 Document (*.doc).")
else:
    print("LibreOffice was unavailable. Open the saved .docx in desktop Word and save it as Word 97–2003 Document (*.doc).")


print(f"Saved the report, CSVs and two 600 dpi JPEGs in {OUTPUT.resolve()}.")